# 02 — Modelo No Supervisado: Detección de Anomalías Transaccionales
**Proyecto Final ADM e IA 2026-2 · Detección de Fraude**

**Responsable:** Orlando Lomán Córdova

**Dataset:** Credit Card Fraud Detection (Kaggle) — `creditcard.csv`

**Modelos:** DBSCAN (detección de anomalías) + GMM (perfilamiento de clusters)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)

## 1. Carga y Exploración Inicial

In [ ]:
# cargar desde la ruta relativa del repositorio
df = pd.read_csv('../data/raw/creditcard.csv')

print(f'dimensiones: {df.shape}')
print(f'\ndistribución de clase (0=normal, 1=fraude):')
print(df['Class'].value_counts())
print(f'\nproporción de fraude: {df["Class"].mean():.4%}')
print(f'\nvalores faltantes: {df.isnull().sum().sum()}')
df.describe().round(2)

## 2. Preprocesamiento

Las variables V1–V28 ya vienen transformadas por PCA (anonimizadas). Solo
necesitamos escalar `Amount` y `Time` para que estén en la misma magnitud.
La columna `Class` **no se usa en el entrenamiento** (modelo no supervisado);
solo se reserva para validar los resultados al final.

In [ ]:
# separar etiqueta real (solo para validación posterior)
y_true = df['Class'].values
X = df.drop(columns=['Class']).copy()

# escalar Amount y Time (V1-V28 ya están en escala PCA)
scaler = StandardScaler()
X[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

print('variables escaladas: Amount, Time')
print(f'matriz de features: {X.shape}')

## 3. Submuestreo Estratificado

DBSCAN tiene complejidad $O(n^2)$ en el peor caso. Con 285 k filas el tiempo
de cómputo sería prohibitivo. Se toma una muestra estratificada de **8 000 filas**
manteniendo la proporción original de fraudes para no sesgar el análisis.

In [ ]:
N_MUESTRA = 8000

idx_normal = np.where(y_true == 0)[0]
idx_fraude = np.where(y_true == 1)[0]

# mantener proporción original ~0.17% fraude
n_fraude  = int(N_MUESTRA * 0.002)   # ~16 fraudes reales en la muestra
n_normal  = N_MUESTRA - n_fraude

idx_sel = np.concatenate([
    np.random.choice(idx_normal, n_normal, replace=False),
    np.random.choice(idx_fraude, n_fraude, replace=False)
])
np.random.shuffle(idx_sel)

X_sub  = X.iloc[idx_sel].values
y_sub  = y_true[idx_sel]

print(f'muestra: {X_sub.shape[0]} filas  |  fraudes reales en muestra: {y_sub.sum()}')

## 4. Reducción de Dimensionalidad con PCA (2D para visualización)

Reducimos las 30 variables a 2 componentes principales para poder graficar
los clusters y anomalías en un plano cartesiano.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_sub)

var_exp = pca.explained_variance_ratio_
print(f'varianza explicada: PC1={var_exp[0]:.2%}  PC2={var_exp[1]:.2%}')
print(f'varianza total conservada: {sum(var_exp):.2%}')

# grafica de los datos en 2D antes de clustering
plt.figure(figsize=(8, 5))
plt.scatter(X_2d[y_sub==0, 0], X_2d[y_sub==0, 1],
            s=4, alpha=0.3, color='steelblue', label='normal')
plt.scatter(X_2d[y_sub==1, 0], X_2d[y_sub==1, 1],
            s=30, alpha=0.9, color='red', label='fraude real', marker='*')
plt.title('transacciones proyectadas con PCA (2D)')
plt.xlabel(f'PC1 ({var_exp[0]:.1%})')
plt.ylabel(f'PC2 ({var_exp[1]:.1%})')
plt.legend()
plt.tight_layout()
plt.savefig('../results/pca_vista_inicial.png', dpi=130)
plt.show()

## 5. Modelo 1 — DBSCAN (Detección de Anomalías)

DBSCAN clasifica cada punto como:
- **Punto núcleo:** tiene al menos `min_samples` vecinos dentro del radio `eps`.
- **Punto frontera:** está dentro del radio de un punto núcleo pero no tiene suficientes vecinos propios.
- **Ruido (etiqueta -1):** no pertenece a ningún cluster → candidato a anomalía / fraude.

In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=10, n_jobs=-1)
labels_db = dbscan.fit_predict(X_2d)

n_clusters  = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_anomalias = np.sum(labels_db == -1)
prop_anom   = n_anomalias / len(labels_db)

print(f'clusters encontrados: {n_clusters}')
print(f'anomalías detectadas: {n_anomalias}  ({prop_anom:.2%} de la muestra)')

# silueta (excluyendo ruido)
mask_valid = labels_db != -1
if mask_valid.sum() > 1 and len(set(labels_db[mask_valid])) > 1:
    sil = silhouette_score(X_2d[mask_valid], labels_db[mask_valid])
    print(f'silhouette score (sin ruido): {sil:.4f}')

# grafica DBSCAN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# panel izquierdo: clusters
colores = plt.cm.tab10(labels_db / max(labels_db.max(), 1))
colores[labels_db == -1] = [0.8, 0.0, 0.0, 1.0]   # rojo para anomalías
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=colores, s=5, alpha=0.5)
axes[0].set_title(f'DBSCAN — {n_clusters} clusters + anomalías (rojo)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

# panel derecho: anomalías vs fraudes reales
anomalia_pred = (labels_db == -1).astype(int)
colores2 = np.where(
    (anomalia_pred == 1) & (y_sub == 1), 'red',
    np.where(anomalia_pred == 1, 'orange',
    np.where(y_sub == 1, 'purple', 'steelblue'))
)
for color, label in zip(['steelblue','orange','red','purple'],
                        ['normal','falso positivo','TP (fraude detectado)','fraude no detectado']):
    mask = colores2 == color
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=color, s=8 if color=='steelblue' else 40,
                    alpha=0.5 if color=='steelblue' else 0.9, label=label)
axes[1].set_title('anomalías DBSCAN vs fraudes reales')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/anomalias.png', dpi=130)
plt.show()

In [ ]:
# validacion contra etiquetas reales (solo para analisis, no uso en entrenamiento)
print('=== validación DBSCAN vs etiquetas reales ===')
print(confusion_matrix(y_sub, anomalia_pred))
print(classification_report(y_sub, anomalia_pred,
                             target_names=['normal', 'fraude']))

## 6. Modelo 2 — GMM (Perfilamiento de Clusters)

El Modelo de Mezcla Gaussiana (GMM) asume que los datos provienen de $K$
distribuciones gaussianas. Cada transacción recibe una probabilidad de
pertenencia a cada componente. Las transacciones con **log-verosimilitud
muy baja bajo todos los componentes** son candidatas a fraude: no se
parecen a ningún perfil de transacción normal.

In [ ]:
# seleccionar K optimo con BIC
bics = []
ks   = range(2, 9)
for k in ks:
    gm = GaussianMixture(n_components=k, covariance_type='full',
                         random_state=SEED)
    gm.fit(X_2d)
    bics.append(gm.bic(X_2d))

k_opt = ks[np.argmin(bics)]
print(f'K optimo segun BIC: {k_opt}')

plt.figure(figsize=(6, 3))
plt.plot(list(ks), bics, 'o-', color='steelblue')
plt.axvline(k_opt, color='red', linestyle='--', label=f'K={k_opt}')
plt.title('criterio BIC para seleccionar K en GMM')
plt.xlabel('numero de componentes K')
plt.ylabel('BIC (menor = mejor)')
plt.legend(); plt.tight_layout()
plt.savefig('../results/gmm_bic.png', dpi=130)
plt.show()

In [ ]:
# entrenar GMM con K optimo
gmm = GaussianMixture(n_components=k_opt, covariance_type='full',
                      random_state=SEED)
gmm.fit(X_2d)

labels_gmm   = gmm.predict(X_2d)
log_prob     = gmm.score_samples(X_2d)   # log-verosimilitud por punto

# umbral: percentil 1 de log-probabilidad = anomalias
umbral_gmm    = np.percentile(log_prob, 1)
anomalia_gmm  = (log_prob < umbral_gmm).astype(int)

sil_gmm = silhouette_score(X_2d, labels_gmm)
print(f'silhouette score GMM: {sil_gmm:.4f}')
print(f'umbral log-prob: {umbral_gmm:.4f}')
print(f'anomalias GMM detectadas: {anomalia_gmm.sum()}  ({anomalia_gmm.mean():.2%})')

# grafica GMM
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# clusters GMM
scatter = axes[0].scatter(X_2d[:, 0], X_2d[:, 1],
                           c=labels_gmm, cmap='tab10', s=5, alpha=0.5)
axes[0].set_title(f'GMM — {k_opt} componentes')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[0])

# log-verosimilitud como mapa de calor
sc = axes[1].scatter(X_2d[:, 0], X_2d[:, 1],
                     c=log_prob, cmap='RdYlGn', s=5, alpha=0.6)
axes[1].scatter(X_2d[anomalia_gmm==1, 0], X_2d[anomalia_gmm==1, 1],
                s=40, color='black', marker='x', label='anomalia GMM')
axes[1].set_title('log-verosimilitud GMM (rojo = bajo = sospechoso)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(fontsize=8)
plt.colorbar(sc, ax=axes[1])

plt.tight_layout()
plt.savefig('../results/clusters.png', dpi=130)
plt.show()

print('\n=== validación GMM vs etiquetas reales ===')
print(confusion_matrix(y_sub, anomalia_gmm))
print(classification_report(y_sub, anomalia_gmm,
                             target_names=['normal', 'fraude']))

## 7. Comparación y Métricas Finales

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score

metricas = pd.DataFrame({
    'modelo': ['DBSCAN', 'GMM'],
    'anomalias_detectadas': [anomalia_pred.sum(), anomalia_gmm.sum()],
    'proporcion': [anomalia_pred.mean(), anomalia_gmm.mean()],
    'precision': [
        precision_score(y_sub, anomalia_pred, zero_division=0),
        precision_score(y_sub, anomalia_gmm,  zero_division=0)
    ],
    'recall': [
        recall_score(y_sub, anomalia_pred, zero_division=0),
        recall_score(y_sub, anomalia_gmm,  zero_division=0)
    ],
    'f1': [
        f1_score(y_sub, anomalia_pred, zero_division=0),
        f1_score(y_sub, anomalia_gmm,  zero_division=0)
    ]
})

print('=== TABLA COMPARATIVA ===')
print(metricas.to_string(index=False))

# guardar metricas para integracion con el equipo
metricas.to_csv('../results/metricas_no_supervisado.csv', index=False)
print('\nguardado: metricas_no_supervisado.csv')

## 8. Interpretación de Resultados

**DBSCAN** identifica anomalías estructurales: transacciones que no caben en
ninguna zona de alta densidad del espacio de características. Estas son
las más alejadas del comportamiento habitual, independientemente de su
clase real.

**GMM** modela el comportamiento normal con $K$ perfiles gaussianos. Una
transacción con log-verosimilitud muy baja bajo todos los perfiles no se
parece a ningún tipo de transacción legítima conocida, lo que la hace
sospechosa.

**Ambos modelos trabajan sin etiquetas**: el recall que se reporta es solo
una validación a posteriori para cuantificar cuántos fraudes reales
coincidieron con las anomalías detectadas. En producción, las etiquetas
no estarían disponibles.

**Limitaciones:**
- DBSCAN es sensible a los hiperparámetros `eps` y `min_samples`; un
  cambio pequeño puede alterar significativamente el número de anomalías.
- GMM asume distribuciones gaussianas; si los clusters tienen formas
  irregulares, el ajuste puede ser pobre.
- Ambos modelos trabajan sobre la proyección PCA de 2 componentes para
  visualización; el modelo de producción debería usar más componentes.

**Salida hacia el pipeline:** el vector `anomalia_pred` (DBSCAN) y
`log_prob` (GMM) se pasan como features adicionales al modelo supervisado
de Andrés para enriquecer la predicción.

In [ ]:
# exportar features para el modelo supervisado
df_features_out = pd.DataFrame({
    'idx_original': idx_sel,
    'dbscan_anomalia': anomalia_pred,
    'gmm_logprob': log_prob,
    'gmm_cluster': labels_gmm,
    'Class': y_sub
})
df_features_out.to_csv('../data/processed/features_no_supervisado.csv', index=False)
print('guardado: features_no_supervisado.csv')
print(f'filas exportadas: {len(df_features_out)}')
df_features_out.head()